Import Library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from PIL import Image
import json
import os
import random
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import re
from tqdm.auto import tqdm 
import requests
import zipfile
import pickle

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Dành cho Mac
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

Using device: mps


In [2]:
COCO_VAL_IMAGES_URL = "http://images.cocodataset.org/zips/val2014.zip"
VQA_QUESTIONS_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip"
VQA_ANNOTATIONS_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip"

def download_file(url, save_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024  # 1 Kibibyte
    
    print(f"Downloading {os.path.basename(save_path)}...")
    with open(save_path, 'wb') as file, tqdm(
        desc=os.path.basename(save_path),
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(block_size):
            size = file.write(data)
            bar.update(size)

def setup_data():
    #Tạo thư mục data
    base_dir = os.getcwd()
    data_dir = os.path.join(base_dir, 'data')
    img_root = os.path.join(data_dir, 'images')
    
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    if not os.path.exists(img_root):
        os.makedirs(img_root)

    print(f"Data Folder: {data_dir}")

    #XỬ LÝ ẢNH
    val2014_dir = os.path.join(img_root, 'val2014')
    zip_path = os.path.join(data_dir, 'val2014.zip')

    if os.path.exists(val2014_dir) and len(os.listdir(val2014_dir)) > 0:
        print("Image existed")
    else:
        if not os.path.exists(zip_path):
            print("Dowloading")
            download_file(COCO_VAL_IMAGES_URL, zip_path)
        
        # Giải nén
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(img_root)
        
        
    #XỬ LÝ CÂU HỎI (QUESTIONS)
    target_json = "v2_OpenEnded_mscoco_val2014_questions.json"
    json_path = os.path.join(data_dir, target_json)
    q_zip_path = os.path.join(data_dir, "questions.zip")

    if os.path.exists(json_path):
        print("Question existed")
    else:
        print("Downloading...")
        download_file(VQA_QUESTIONS_URL, q_zip_path)
        
        print("Zip processing...")
        with zipfile.ZipFile(q_zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)

    final_img_dir = val2014_dir if os.path.exists(val2014_dir) else img_root

    #XỬ LÝ ĐÁP ÁN (ANNOTATIONS)
    target_a_json = "v2_mscoco_val2014_annotations.json"
    anno_path = os.path.join(data_dir, target_a_json)
    a_zip_path = os.path.join(data_dir, "annotations.zip")

    if os.path.exists(anno_path):
        print("Annotation file existed")
    else:
        download_file(VQA_ANNOTATIONS_URL, a_zip_path)
        print("Extracting annotations...")
        with zipfile.ZipFile(a_zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)

    # Xác định đường dẫn ảnh cuối cùng
    final_img_dir = val2014_dir if os.path.exists(val2014_dir) else img_root
    
    # Trả về Ảnh, Câu hỏi, Đáp án
    return final_img_dir, json_path, anno_path

try:
    IMG_DIR, JSON_FILE, ANNO_FILE = setup_data()

    print("-" * 30)
    print("Processing complete")
    print(f"Image Dir : {IMG_DIR}")
    print(f"Question  : {JSON_FILE}")
    print(f"Annotation: {ANNO_FILE}")
except Exception as e:
    print(f"Error: {e}")

Data Folder: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data
Image existed
Question existed
Annotation file existed
------------------------------
Processing complete
Image Dir : /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/images/val2014
Question  : /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/v2_OpenEnded_mscoco_val2014_questions.json
Annotation: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/v2_mscoco_val2014_annotations.json


In [3]:
class Vocabulary:
    def __init__(self):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = 1

    def __len__(self): return len(self.itos)

    @staticmethod
    def tokenizer(text):
        # Chuyển về chữ thường, bỏ dấu câu cơ bản
        return str(text).lower().replace('?', '').replace('.', '').replace(',', '').split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1
        
        # Chỉ thêm từ xuất hiện nhiều hơn threshold
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
    
    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [4]:
class VQADataset(Dataset):
    def __init__(self, json_path, img_dir, annotation_path=None, vocab=None, transform=None, top_k=1000):
        self.img_dir = img_dir
        self.transform = transform
        
        # Load dữ liệu câu hỏi (JSON)
        print(f"Processing Questions...: {os.path.basename(json_path)}...")
        with open(json_path, 'r') as f:
            data_raw = json.load(f)
            self.data = data_raw['questions'] if 'questions' in data_raw else data_raw

        #Xử lý file Đáp án (Annotations)
        self.ans_to_idx = {}
        self.qid_to_ans = {}
        
        if annotation_path:
            print(f"Processing Annotations...: {os.path.basename(annotation_path)}...")
            with open(annotation_path, 'r') as f:
                anno_data = json.load(f)['annotations']
            
            #Map question_id -> đáp án
            self.qid_to_ans = {item['question_id']: item['multiple_choice_answer'] for item in anno_data}
            
            #Lọc Top K đáp án phổ biến nhất để làm nhãn (Label)
            all_answers = [item['multiple_choice_answer'] for item in anno_data]
            counter = Counter(all_answers)
            top_answers = counter.most_common(top_k)
            
            #"yes" = 0, "no" = 1
            self.ans_to_idx = {ans: i for i, (ans, count) in enumerate(top_answers)}
            print(f"Found {len(self.ans_to_idx)} common answers as labels.")

        # Tự động xây dựng từ điển câu hỏi nếu chưa có
        if vocab is None:
            print("Processing Vocab...")

            self.vocab = Vocabulary() 
            all_questions = [item.get('question', '') for item in self.data]
            self.vocab.build_vocabulary(all_questions)
            print(f"Vocab size: {len(self.vocab)} words.")
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        
        #XỬ LÝ ẢNH 
        img_id = item['image_id']
        img_filename = f"COCO_val2014_{int(img_id):012d}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            image = Image.new('RGB', (224, 224)) # Ảnh đen dự phòng
            
        if self.transform:
            image = self.transform(image)

        #XỬ LÝ CÂU HỎI
        question = item.get('question', '')
        q_vec = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(question) + [self.vocab.stoi["<EOS>"]]
        
        #XỬ LÝ NHÃN (LABEL)
        label = -1 # Mặc định là -1 (ignore index) nếu không tìm thấy
        q_id = item['question_id']
        
        # Tìm đáp án tương ứng với câu hỏi này
        if q_id in self.qid_to_ans:
            ans_text = self.qid_to_ans[q_id]
            # Chỉ lấy label nếu đáp án nằm trong Top K
            if ans_text in self.ans_to_idx:
                label = self.ans_to_idx[ans_text]
        
        return image, torch.tensor(q_vec), torch.tensor(label, dtype=torch.long)

In [5]:
def collate_fn(batch):
    images, questions, labels = zip(*batch)

    images = torch.stack(images, 0)

    lengths = [len(q) for q in questions]
    questions = pad_sequence(questions, batch_first=True, padding_value=0)

    labels = torch.stack(labels)

    return images, questions, torch.tensor(lengths), labels


In [6]:
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

if 'IMG_DIR' in globals() and 'JSON_FILE' in globals() and 'ANNO_FILE' in globals():
    # Đường dẫn lưu dataset đã chia
    processed_dir = os.path.join(os.path.dirname(JSON_FILE), 'processed')
    if not os.path.exists(processed_dir):
        os.makedirs(processed_dir)
    
    split_file = os.path.join(processed_dir, 'dataset_splits.pkl')
    vocab_file = os.path.join(processed_dir, 'vocab.pkl')
    
    # Kiểm tra xem đã có file lưu chưa
    if os.path.exists(split_file) and os.path.exists(vocab_file):
        print("Loading saved dataset splits...")
        
        # Load vocab
        with open(vocab_file, 'rb') as f:
            vocab = pickle.load(f)
        
        # Load full dataset với vocab đã lưu
        full_dataset = VQADataset(JSON_FILE, IMG_DIR, annotation_path=ANNO_FILE, vocab=vocab, transform=transform)        
        # Load indices đã chia
        with open(split_file, 'rb') as f:
            split_data = pickle.load(f)
            train_indices = split_data['train_indices']
            val_indices = split_data['val_indices']
            test_indices = split_data['test_indices']
        
        # Tạo lại các Subset với indices đã lưu
        from torch.utils.data import Subset
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)
        test_dataset = Subset(full_dataset, test_indices)
        
        print("Loaded from saved files")
    else:
        print("Creating new dataset splits...")
        
        # Load toàn bộ dữ liệu
        full_dataset = VQADataset(JSON_FILE, IMG_DIR, annotation_path=ANNO_FILE, transform=transform)
        
        # (80% Train - 10% Val - 10% Test)
        total_count = len(full_dataset)
        train_size = int(0.8 * total_count)
        val_size = int(0.1 * total_count)
        test_size = total_count - train_size - val_size
        
        # Chia dataset
        train_dataset, val_dataset, test_dataset = random_split(
            full_dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(42)  # Để reproducible
        )
        
        # Lưu vocab
        with open(vocab_file, 'wb') as f:
            pickle.dump(full_dataset.vocab, f)
        
        # Lưu indices của mỗi split
        split_data = {
            'train_indices': train_dataset.indices,
            'val_indices': val_dataset.indices,
            'test_indices': test_dataset.indices
        }
        with open(split_file, 'wb') as f:
            pickle.dump(split_data, f)
        
        print(f"Saved splits to {processed_dir}")
    
    # Tạo DataLoader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

    # In thông tin
    print("-" * 30)
    print(f"Vocab Size: {len(full_dataset.vocab)}")
    print(f"Total Data: {len(full_dataset)}")
    print(f"Train Set : {len(train_dataset)}")
    print(f"Val Set   : {len(val_dataset)}")
    print(f"Test Set  : {len(test_dataset)}")


Loading saved dataset splits...
Processing Questions...: v2_OpenEnded_mscoco_val2014_questions.json...
Processing Annotations...: v2_mscoco_val2014_annotations.json...
Found 1000 common answers as labels.
Loaded from saved files
------------------------------
Vocab Size: 11074
Total Data: 214354
Train Set : 171483
Val Set   : 21435
Test Set  : 21436


In [18]:
class VQAModel(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_size=512, hidden_size=512, num_layers=1):
        super(VQAModel, self).__init__()

        # ================= IMAGE ENCODER =================
        resnet = models.resnet18(pretrained=True)
        resnet.fc = nn.Identity()  # bỏ lớp FC cuối
        self.cnn = resnet
        image_feature_dim = 512  # resnet18 output

        # ================= TEXT ENCODER =================
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # ================= CLASSIFIER =================
        self.fc = nn.Linear(image_feature_dim + hidden_size, num_classes)

    def forward(self, images, questions, lengths):

        # ===== IMAGE =====
        image_features = self.cnn(images)  # (B, 512)

        # ===== TEXT =====
        embedded = self.embedding(questions)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        text_features = hidden[-1]  # (B, hidden_size)

        # ===== FUSION =====
        combined = torch.cat((image_features, text_features), dim=1)

        output = self.fc(combined)

        return output


In [19]:
model = VQAModel(
    vocab_size=len(full_dataset.vocab),
    num_classes=len(full_dataset.ans_to_idx),
    embed_size=512,
    hidden_size=512,
    num_layers=1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 15

def calculate_accuracy(output, labels):
    preds = torch.argmax(output, dim=1)
    
    mask = labels != -1  # bỏ label ignore
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    
    return correct, total

In [21]:
best_val_acc = 0

for epoch in range(EPOCHS):
    
    # TRAIN
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]")

    for images, questions, lengths, labels in train_bar:
        images = images.to(device)
        questions = questions.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images, questions, lengths)
        loss = criterion(output, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        train_loss += loss.item()

        correct, total = calculate_accuracy(output, labels)
        train_correct += correct
        train_total += total

        train_bar.set_postfix(loss=loss.item())

    train_acc = train_correct / train_total if train_total > 0 else 0
    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]")

    with torch.no_grad():
        for images, questions, lengths, labels in val_bar:
            images = images.to(device)
            questions = questions.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            output = model(images, questions, lengths)
            loss = criterion(output, labels)

            val_loss += loss.item()

            correct, total = calculate_accuracy(output, labels)
            val_correct += correct
            val_total += total

            val_bar.set_postfix(loss=loss.item())

    val_acc = val_correct / val_total if val_total > 0 else 0
    val_loss /= len(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vqa_model.pth")
        print("Saved Best Model")

print("Training finished!")


Epoch 1/15 [TRAIN]:   0%|          | 0/5359 [00:00<?, ?it/s]

KeyboardInterrupt: 